In [1]:
import os, sys, json
import pandas as pd
from google.cloud import bigquery
PROJECT_ID = "project-5eb4889f-74fb-4ad5-a76"
LOCATION   = "EU"

creds = None

if "google.colab" in sys.modules:
    from google.colab import auth
    auth.authenticate_user()
elif os.environ.get("GCP_SA_KEY"):
    from google.oauth2 import service_account
    creds = service_account.Credentials.from_service_account_info(
        json.loads(os.environ["GCP_SA_KEY"]),
        scopes=["https://www.googleapis.com/auth/cloud-platform"])

client = bigquery.Client(project=PROJECT_ID, location=LOCATION, credentials=creds)
print("проєкт:", client.project)

проєкт: project-5eb4889f-74fb-4ad5-a76


In [2]:
ds_raw = bigquery.Dataset(f"{PROJECT_ID}.nbu_raw")
ds_raw.location = "EU"
client.create_dataset(ds_raw, exists_ok=True)

ds_dwh = bigquery.Dataset(f"{PROJECT_ID}.nbu_dwh")
ds_dwh.location = "EU"
client.create_dataset(ds_dwh, exists_ok=True)

print("датасети готові: nbu_raw, nbu_dwh")

датасети готові: nbu_raw, nbu_dwh


In [3]:
TABLE_ID = f"{PROJECT_ID}.nbu_raw.raw_rates"

schema = [
    bigquery.SchemaField("ingested_at",   "TIMESTAMP"),
    bigquery.SchemaField("business_date", "DATE"),
    bigquery.SchemaField("request_url",   "STRING"),
    bigquery.SchemaField("payload",       "STRING"),
    bigquery.SchemaField("payload_hash",  "STRING"),
]

table = bigquery.Table(TABLE_ID, schema=schema)
table.time_partitioning = bigquery.TimePartitioning(field="business_date")
client.create_table(table, exists_ok=True)

print("таблиця готова:", TABLE_ID)

таблиця готова: project-5eb4889f-74fb-4ad5-a76.nbu_raw.raw_rates


In [4]:
import requests

URL = "https://bank.gov.ua/NBUStatService/v1/statdirectory/exchange?json"

resp = requests.get(URL, timeout=30)
resp.raise_for_status()
data = resp.json()

print(len(data), "валют")
print(data[0])

45 валют
{'r030': 12, 'txt': 'Алжирський динар', 'rate': 0.33508, 'cc': 'DZD', 'exchangedate': '29.09.2026', 'special': None}


In [5]:
import hashlib
import datetime

rows = []
ingested_at = pd.Timestamp.now(tz="UTC")

for item in data:
    payload = json.dumps(item, ensure_ascii=False, sort_keys=True)
    payload_hash = hashlib.md5(payload.encode("utf-8")).hexdigest()
    business_date = datetime.datetime.strptime(item["exchangedate"], "%d.%m.%Y").date()

    rows.append({
        "ingested_at": ingested_at,
        "business_date": business_date,
        "request_url": URL,
        "payload": payload,
        "payload_hash": payload_hash,
    })

df = pd.DataFrame(rows)
print(df.shape)
df.head()

(45, 5)


,ingested_at,business_date,request_url,payload,payload_hash
0,2026-09-28 19:19:34.931998+00:00,2026-09-29,https://bank.gov.ua/NBUStatService/v1/statdire...,"{""cc"": ""DZD"", ""exchangedate"": ""29.09.2026"", ""r...",ce129885992b12cb8981d37769d04f34
1,2026-09-28 19:19:34.931998+00:00,2026-09-29,https://bank.gov.ua/NBUStatService/v1/statdire...,"{""cc"": ""AUD"", ""exchangedate"": ""29.09.2026"", ""r...",5a009d0efe2919200e161d7729e92a87
2,2026-09-28 19:19:34.931998+00:00,2026-09-29,https://bank.gov.ua/NBUStatService/v1/statdire...,"{""cc"": ""BDT"", ""exchangedate"": ""29.09.2026"", ""r...",249893b7f01f9e3f757c75fff9e22db2
3,2026-09-28 19:19:34.931998+00:00,2026-09-29,https://bank.gov.ua/NBUStatService/v1/statdire...,"{""cc"": ""CAD"", ""exchangedate"": ""29.09.2026"", ""r...",e5c7be22adb1d5e365aeb3f4527c7c1f
4,2026-09-28 19:19:34.931998+00:00,2026-09-29,https://bank.gov.ua/NBUStatService/v1/statdire...,"{""cc"": ""CNY"", ""exchangedate"": ""29.09.2026"", ""r...",83e5109caf1f055738886e4dc1643216


In [6]:
cfg = bigquery.LoadJobConfig(schema=schema, write_disposition="WRITE_APPEND")
client.load_table_from_dataframe(df, TABLE_ID, job_config=cfg).result()

print("записано рядків:", len(df))

/Users/ik/nbu-pipeline/.venv/lib/python3.13/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


записано рядків: 45


In [7]:
# після двох запусків ноутбука rows_cnt за дату подвоюється, а uniq_cnt лишається тим самим. запис іде в режимі WRITE_APPEND: кожен запуск дописує нову порцію рядків і нічого не видаляє, тому журнал завантажень зберігає обидва запуски. uniq_cnt не змінюється, бо payload_hash рахується від вмісту об'єкта: курс не змінився між запусками, хеші другої порції збігаються з першою, тож унікальних значень стільки ж, скільки валют. Це не помилка, а нормальна поведінка bronze.

check_sql = f"""
SELECT business_date, COUNT(*) AS rows_cnt, COUNT(DISTINCT payload_hash) AS uniq_cnt
FROM `{PROJECT_ID}.nbu_raw.raw_rates`
GROUP BY business_date
ORDER BY business_date
"""

check_df = client.query(check_sql).to_dataframe()
check_df

/Users/ik/nbu-pipeline/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,business_date,rows_cnt,uniq_cnt
0,2026-09-23,45,45
1,2026-09-25,90,45
2,2026-09-29,135,45
